# M1P3 - Use of Programs in Bash

In this notebook installing packages and using programs in bash will be covered. 


## In this Practice: 
* [Packages vs Programs](#pro)
* [How to Install Packages to a Conda Enviornment](#conda)
* [Utilizing programs within a bash script](#script)
* [Retriving files with `wget`](#wget)
* [Utilizing the `blast` Package](#blast)

```{note}
**Runs on an HPC.** This notebook was written for the Hellbender HPC cluster (University of Missouri) and uses SLURM and/or large sequencing files. It is shown here for reading; saved outputs were removed for privacy. Set `REPO_DIR`, `DATA_DIR` and `WORK_DIR` (see the repository README) and download inputs with `data/fetch_data.sh` before running it on your own cluster.
```

# Packages vs Programs
### Packages
In M1P1, we talked about [conda enviornments](https://docs.conda.io/projects/conda/en/latest/user-guide/tasks/manage-environments.html). Within a conda environment [packages](https://docs.conda.io/projects/conda/en/stable/user-guide/concepts/packages.html), or a collection of files, programs, and dependencies, are installed and isolated from the rest of a computational system. Packages typically contain one or more programs, along with their associated files, libraries, and configuration files. Packages are organized so that they can be easily downloaded from a repository and installed. For example, if you wanted to install the `blast` package, you could run `conda install -c bioconda blast` within your desired environment. The `blast` package contains several programs such as `makeblastdb`, `blastn`, `blastp`, and `blastdbcmd`.

### Programs
A program is an executable file that is run within the shell and can be contained, along with its files and dependencies, within an environment. A program performs a specific job (for example, creating a database, aligning sequences, etc). A program is run by typing its name into the environment. For example, if you wanted to create a blast database from a `fasta` file, you would use the command `makeblastdb -in {$myFasta}.fa -dbtype nucl -parse_seqids -out "$DB"`. This command runs the program `makeblastdb` with the parameters `-dbtype`, `-parse_seqids`, and `-out`. 

### Parameters
A [parameter](https://www.baeldung.com/linux/use-command-line-arguments-in-bash-script) is an option or input that is provided alongside a program command to control how the program is run. Parameters tell the program what to do or how to do it. [There are three main types of parameters](https://www.gnu.org/software/bash/manual/html_node/Shell-Parameters.html): 
* Flags: Options that turn things on/off (e.g., `--verbose`, `-l`)
* Key-value pairs: Options that take a value, the value is usually input one space after the parameter(e.g., -in {$filename}, -evalue 1e-5)
* Positional arguments: providing the file after the program name, or a number in a correct predetermined spot.

**IMPORTANT:** This is critical for the versatile use of programs within the command line and can be used to customize the output of a program according to the user's needs. Most programs will output a help menu when the `-h` or `--help` parameter is added after the program name, such as `blast --help` or `makeblastdb -h`. 

Most modern command line tools do not require a specific order for parameters as long as the parameter is correctly written with its flag. In the case that you are using a program that has a parameter that is a positional argument, the order is important.

For example, when using the `cp` program to copy files, the first parameter is the source file and the second parameter is the destination. The command `cp /Module_1/Lab/M1P4_Use_of_Programs_In_Bash.ipnby /Module_2/Lab/` would copy this notebook to the `Module_2/Lab` folder, but the command `cp /Module_2/Lab/  /Module_1/Lab/M1P4_Use_of_Programs_In_Bash.ipynb` would return an error because the destination must be a folder and not a file type. (Folders can be copied using the `-r` parameter, but that is negligible in our example)

# How to Install Packages in a Conda Environment
<u>**NOTE:** Conda is down on Hellbender as of 10/14/25, and therefore, any time we see commands using `conda` within the syntax, we are replacing that with `mamba`. Remember to use `module load mamba` each time you open a terminal or run a bash script that requires the use of the `mamba` module. </u>

To install packages to a conda environment requires a couple of things: 
1.  Conda must be installed, and in the case of our HPC cluster, the conda module must be loaded with the command `module load miniconda3`
    * You can check if conda is active the same way you can check if any program is installed or active, by typing its name into the command line. In this case, typing `conda` into the terminal should return the version of conda that is active.
2. The environment to which the package will be installed must be activated. In this example, we want to install the `blast` package, which contains many alignment and database tools developed by NCBI's BLAST, into our environment `Module_1`. To activate `Module_1`, we will use `source activate Module_1` in the command line
    * Note: Our HPC cluster uses a deprecated term, `source`, in place of `conda` when activating or deactivating enviornments. This is due to the way dependencies are managed, and the use of `conda activate {$MyEnv}` will not function (generally) in its place. On other systems, `conda activate` is a valid command.

Once these two conditions are met —Conda is available and you are working within your desired environment —you can install packages simply with the command `conda install {$packageName}`. There are many parameters you can use, and there are references to these in [M1P1](M1P1_Creating_and_Managing_Environments.ipynb). The most common parameter is `-c` for naming the [channel](https://docs.conda.io/projects/conda/en/stable/user-guide/concepts/channels.html) from which to download a package. The most commonly used channels are `-c default`, `-c forge`, and `-c bioconda`.

Please follow along within the terminal to install the `blast` package to your `Module_1' environment, as you will need it to complete [M1E1](../exercise/M1E1_Blast_Bash_Scripting.ipynb). 

Type the following into a new terminal line by line, accessed from the Jupyter Home page: 

`module load mamba`

`source activate Module_1`

`mamba install -c bioconda blast`
    
As the program installs, ensure to enter `y` or `yes` to any options that occur. Once the package is installed, you can verify the installation with `which blastn` or `blastn -version` from the command line. To see which packages are installed within your environment, you can use the `conda list` command. Alternatively, to check a different environment, use the `conda env list | grep {$myEnvName}` command.

**PRESS `SHIFT + ENTER` TO WATCH M1P3V6 BELOW ON HOW TO RUN A BASH SCRIPT. PLEASE USE THE PYTHON3 KERNEL FOR HTML MAGIC.**

#M1P3V6
<iframe width="560" height="315" src="https://www.youtube.com/embed/1rGU0fhXzYI?si=hkT_lzohGvHxEWlC" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

# Utilizing Programs Within a Bash Script

Programs are run by typing their name into the command line, as we had mentioned previously. Each time you use the commands `mv`, `cp`, or `mkdir`, you are running a program. For programs installed from packages within your conda environment, they are run the same way. Programs can be run within bash scripts in the same fashion, by typing their name and parameters into a line of code within the bash script. Programs can even be run within loops or conditionally. 

The most important part of learning with the command line is the ability to read documentation and access help from external resources to understand code, commands, and programs. Most programs have a built-in "help page" which contains a list of basic parameters as well as a link to their documentation. We can access these by running the program with the parameter `-h` or `--help`. In this module's exercise, you will be asked to run several programs from the `blast` package, so let's take some time to understand how to decipher the documentation that comes along with these programs. 

For reference in [M1E1](../exercise/M1E1_Blast_Bash_Scripting.ipynb), the documentation for `blast` is [here](https://bioconda.github.io/recipes/blast/README.html)

**Note:** Up until this point, all bash programs have been executed within the `Python 3(ipykernel)` kernel native to Jupyter Notebook using only preinstalled programs. This kernel is not active within the Module_1 environment. Thus if you install the package `blast` to your `Module_1` enviroment the blast package is not installed to your `Python 3(ipykernel)` so running the `!blast --help` command within the cell below will not work. R does not natively support bash magic (the use of `!`, `%`, or `%%bash`), so if you wish to run the `blast` program within the cell below you will have to use the `R (Module_1)` kernel AND use `system("module load miniconda3;source activate Module_1;makeblastdb -h", intern = TRUE)` within the code cell.

To write bash scripts within R you can use the `system()` function, place each command followed by a `;` within quotation marks and add the parameter `intern=TRUE` for the console to print the output. In future modules, the ipykernel will be installed in the environment; however, in Module_1, this allows students to conceptualize how kernels, environments, packages, and programs are isolated.

In [ ]:
#!!!!!!!!!!!!!!!For this to run, you will need to change the kernel to "R (Module_1)"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
system("module load miniconda3;source activate Module_1;makeblastdb -h", intern = TRUE)

# Retrieving files with `wget`
`wget` is a command-line utility used to download files from the internet. It supports downloading via HTTP, HTTPS, and FTP protocols. It is commonly used in Bash scripts and terminal sessions to fetch data or software directly to your local machine or HPC environment.
* Non-interactive downloader: You can start a download and even log out; it continues running.
* Easy to use in scripts: Automates fetching files as part of workflows.

| Command/Option                         | Description                                                  | Example                                                      |
| -------------------------------------- | ------------------------------------------------------------ | ------------------------------------------------------------ |
| `wget URL`                             | Download file from the specified URL                         | `wget https://example.com/file.txt`                          |
| `wget -O filename URL`                 | Download and save file with a custom filename                | `wget -O data.fa.gz https://example.com/file.fa.gz`          |
| `wget -c URL`                          | Continue/resume a partially downloaded file                  | `wget -c https://example.com/largefile.zip`                  |
| `wget -q URL`                          | Quiet mode (no output except errors)                         | `wget -q https://example.com/file.txt`                       |
| `wget --limit-rate=RATE`               | Limit download speed (e.g., 200k for 200 KB/s)               | `wget --limit-rate=200k https://example.com/file.zip`        |
| `wget -r URL`                          | Recursive download (download directories and subdirectories) | `wget -r https://example.com/directory/`                     |
| `wget --no-check-certificate URL`      | Download from HTTPS sites without verifying SSL certificates | `wget --no-check-certificate https://example.com/file`       |
| `wget --user=USER --password=PASS URL` | Download with HTTP authentication                            | `wget --user=admin --password=1234 https://example.com/file` |
| `wget --spider URL`                    | Check if a file exists without downloading                   | `wget --spider https://example.com/file.txt`                 |
| `wget -b URL`                          | Download in the background                                   | `wget -b https://example.com/file.zip`                       |


To run wget in a bash script is simple, and an example downloading the `Arabidopsis_thaliana.TAIR10.dna.chromosome.1.fa.gz` fasta file from the [Ensembl FTP Arabidopsis Website](https://ftp.ensemblgenomes.ebi.ac.uk/pub/plants/release-61/fasta/arabidopsis_thaliana/) is shown below. 

<u>**IMPORTANT FOR THE USE OF THIS CODE CELL BELOW: SWITCH BACK TO THE `Python3 (ipykernel)` TO USE THE BASH MAGIC WITHIN THE CELL**</u>

**NOTE**: This is not the CDNA. For M1E1, please make sure to navigate to the cDNA folder within the Ensemble FTP server and choose the correct link for the file `	Arabidopsis_thaliana.TAIR10.cdna.all.fa.gz`

In [ ]:
%%bash
#MAKE SURE TO CHANGE THE KERNEL BACK TO "Python3 (ipykernel)"

# Check your pwd, as without listing a directory, the -O parameter on wget will save the file to the current working directory.
#In this case, it will save to $WORK_DIR/module_1_bash_hpc/practice/
pwd

# Let's create the name of the file as a variable and call it DNA_FASTA_FILE
DNA_FASTA_FILE="Arabidopsis_thaliana.TAIR10.dna.chromosome.1.fa"
DNA_FASTA_URL="https://ftp.ensemblgenomes.ebi.ac.uk/pub/plants/release-61/fasta/arabidopsis_thaliana/dna/Arabidopsis_thaliana.TAIR10.dna.chromosome.1.fa.gz"

#We use ${VARNAME} to denote the name we want for our file so that it can be used later in the gzip command as well. Note that since the file we are downloading
#is a zipped file (.gz), we do NOT include this in the variable name DNA_FASTA_FILE. 
#This is so we could pipe the variable to another program, such as makeblastdb.
wget -O ${DNA_FASTA_FILE}.gz  $DNA_FASTA_URL

Note: The Arabidopsis Chromosome 1 DNA fasta file was fetched with `wget` as a `.fa.gz` file, and we will need to extract it with the program `gzip` to convert the format to `.fa`. See below for a cheat sheet for `gzip`

In [ ]:
%%bash
#MAKE SURE TO CHANGE THE KERNEL BACK TO "Python3 (ipykernel)"
DNA_FASTA_FILE="Arabidopsis_thaliana.TAIR10.dna.chromosome.1.fa"

gzip -d ${DNA_FASTA_FILE}.gz

echo "done"

| Command                       | Description                                   | Example                              | Notes                                             |
| ----------------------------- | --------------------------------------------- | ------------------------------------ | ------------------------------------------------- |
| `gzip filename`               | Compress a file, replace original with `.gz`  | `gzip myfile.txt`                    | Original `myfile.txt` replaced by `myfile.txt.gz` |
| `gzip -c filename > out.gz`   | Compress and write output to a new file       | `gzip -c myfile.txt > myfile.txt.gz` | Keeps original file, writes compressed output     |
| `gunzip filename.gz`          | Decompress a `.gz` file                       | `gunzip myfile.txt.gz`               | Replaces compressed file with decompressed        |
| `gzip -d filename.gz`         | Decompress (same as `gunzip`)                 | `gzip -d myfile.txt.gz`              |                                                   |
| `gzip -l filename.gz`         | List info about compressed file               | `gzip -l myfile.txt.gz`              | Shows compressed size, uncompressed size, ratio   |
| `gzip -r directory/`          | Recursively compress all files in a directory | `gzip -r ./myfolder`                 | Compresses all files in directory and subdirs     |
| `gzip -t filename.gz`         | Test integrity of compressed file             | `gzip -t myfile.txt.gz`              | Returns exit code 0 if file is OK                 |
| `zcat filename.gz`            | View decompressed contents without unzipping  | `zcat myfile.txt.gz`                 | Outputs contents to stdout                        |
| `zgrep "pattern" filename.gz` | Search compressed file for pattern            | `zgrep "error" logs.gz`              | Like `grep`, but for gzipped files                |
| `gzip -9 filename`            | Compress with maximum compression             | `gzip -9 myfile.txt`                 | Slower compression, smaller size                  |
| `gzip -1 filename`            | Compress with fastest compression             | `gzip -1 myfile.txt`                 | Faster compression, larger size                   |


# Utilizing the `blast` package (Hints for M1E1)
We will use the `blast` package to identify any common genes located on chromosome 1 of the Arabidopsis genome. You will need to make sure you use the environment `Module_1` so that the `blast` package is installed when running from the terminal. 

**Making a blast database**
Initializing Variables for use with `blast` package

    DB="tair10_db"
    GENE_LIST="Common_Genes_Arabidopsis.txt"
    FASTA_FILE="Arabidopsis_thaliana.TAIR10.cdna.all.fa"
    FASTA_URL="ftp://ftp.ensemblgenomes.org/pub/plants/release-50/fasta/arabidopsis_thaliana/cdna/Arabidopsis_thaliana.TAIR10.cdna.all.fa.gz"
    OUTPUT="Common_Genes_Arabidopsis_Sequences.fa"

Retrieve the fasta file from Ensembl

    wget -O ${FASTA_FILE}.gz $FASTA_URL
    gzip -d ${FASTA_FILE}.gz
    
Using the `makeblastdb` command

    makeblastdb -in "$FASTA_FILE" -dbtype nucl -parse_seqids -out "$DB"

Use a for loop to search your database for a fasta entry from a known list of genes.

    while read -r GENE; do
        if [[ -n "$GENE" ]]; then
            echo "Fetching $GENE..."
            blastdbcmd -db "$DB" -entry "$GENE" >> "$OUTPUT"
        fi
    done < "$GENE_LIST"
    
**INSERT M1P4V1**